In [0]:
%pip install shap==0.51.0
%pip install xgboost==3.2.0


In [0]:
%restart_python

In [0]:
# Importing libraries

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import shap
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from mlflow.tracking import MlflowClient

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

TXN_FEATURE_NAMES = [
    "log_amount", "amount_balance_ratio", "transaction_hour",
    "transaction_dayofweek", "merchant_fraud_rate",
    "customer_avg_transaction_amount", "merchant_category_index",
    "device_type_index", "channel_grouped_index", "location_city_grouped_index"
]

# Loading model + theshold

client = MlflowClient()
registered_name = "workspace.ml_layer.Modelo_Fraude_XGB_Transactions"
versions       = client.search_model_versions(f"name='{registered_name}'")
latest_version = max([int(v.version) for v in versions])
xgb_model      = mlflow.xgboost.load_model(f"models:/{registered_name}/{latest_version}")

username   = spark.sql("SELECT current_user()").collect()[0][0]
experiment = mlflow.get_experiment_by_name(f"/Users/{username}/supervised_models")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'XGB_transactions'",
    order_by=["start_time DESC"], max_results=1
)
production_threshold = float(runs.iloc[0]["metrics.best_threshold"])

# Demo threshold (Only for demo)
demo_threshold = 0.50

print(f"   Model v{latest_version} loaded")
print(f"   Production threshold: {production_threshold:.4f}")
print(f"   Demo threshold:       {demo_threshold:.4f}")

# Sampling cases

txn_test = spark.table("workspace.ml_layer.transaction_test_features") \
    .withColumn("is_fraud", col("is_fraud").cast("double")) \
    .withColumn("features_arr", vector_to_array("features"))

fraud_pool = txn_test.filter("is_fraud = 1").limit(500).toPandas()
legit_pool = txn_test.filter("is_fraud = 0").limit(500).toPandas()

fraud_X = np.array(fraud_pool["features_arr"].tolist())
legit_X = np.array(legit_pool["features_arr"].tolist())

fraud_scores_pool = xgb_model.predict_proba(fraud_X)[:, 1]
legit_scores_pool = xgb_model.predict_proba(legit_X)[:, 1]

fraud_sorted = np.argsort(fraud_scores_pool)[::-1]
legit_sorted = np.argsort(legit_scores_pool)

selected_X = np.array([
    fraud_X[fraud_sorted[0]],
    fraud_X[fraud_sorted[len(fraud_sorted)//4]],
    fraud_X[fraud_sorted[len(fraud_sorted)//2]],
    legit_X[legit_sorted[0]],
    legit_X[legit_sorted[5]]
])
true_labels = ["FRAUD", "FRAUD", "FRAUD", "LEGIT", "LEGIT"]

descriptions = [
    "Transacción de $1,899 a la media noche, 5 veces mayor al promedio del cliente, dispositivo secundario",
    "Transaccion de $301 a las 5PM desde dispositivo nuevo",
    "Transaccion de $43 a las 6PM",
    "Transaccion de $1,487 a la media noche, en linea con promedio del cliente",
    "Transaccion de $1.43 tarde en la noche, desde una cuenta de conocida"
]

# Scoring and SHAP reason codes

scores = xgb_model.predict_proba(selected_X)[:, 1]

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(selected_X)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Demo risk tiers based on threshold

def assign_tier(score):
    """Asigns risk tier for fraud transactions."""
    if score >= 0.65:   return "🔴 CRITICAL"
    elif score >= 0.55: return "🟡 HIGH"
    else:               return "🟠 MEDIUM"

action_map = {
    "🔴 CRITICAL": "AUTO-BLOCK",
    "🟡 HIGH"    : "MANUAL REVIEW",
    "🟠 MEDIUM"  : "ENHANCED MONITORING",
    "🟢 LOW"     : "APPROVE"
}

amounts = np.expm1(selected_X[:, 0])

results = []
for i in range(len(selected_X)):
    impacts = sorted(
        zip(TXN_FEATURE_NAMES, shap_values[i], selected_X[i]),
        key=lambda x: abs(x[1]), reverse=True
    )
    reasons = []
    for fname, sval, fval in impacts[:3]:
        arrow = "↑" if sval > 0 else "↓"
        reasons.append(f"{arrow} {fname} ({fval:.2f})")

    # Clasification based on threshold

    model_fraud = scores[i] >= demo_threshold

    if not model_fraud:
        # Legit should be approved
        tier   = "🟢 LOW"
        action = "APPROVE"
    else:
        # Fraud, use tier
        tier   = assign_tier(scores[i])
        action = action_map[tier]

    correct = "✅" if model_fraud == (true_labels[i] == "FRAUD") else "❌"

    results.append({
        "case"        : f"Case {i+1}",
        "description" : descriptions[i],
        "amount_usd"  : round(float(amounts[i]), 2),
        "true_label"  : true_labels[i],
        "fraud_score" : round(float(scores[i]), 4),
        "risk_tier"   : tier,
        "decision"    : action,
        "correct"     : correct,
        "reason_1"    : reasons[0],
        "reason_2"    : reasons[1],
        "reason_3"    : reasons[2]
    })

results_df = pd.DataFrame(results)

# Sorting table

results_df = results_df.sort_values(
    by=["true_label", "fraud_score"],
    ascending=[False, False]
).reset_index(drop=True)

results_df["case"] = [f"Case {i+1}" for i in range(len(results_df))]

# Displaying results and saving data

print("\n" + "="*90)
print("  🛡️  LIVE FRAUD DETECTION DEMO — 5 Real Transactions")
print(f"  (Decision boundary for this demo: {demo_threshold:.2f})")
print("="*90)
display(results_df)

spark.createDataFrame(results_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.demo_results")

print("\nDemo results saved to workspace.ml_layer.demo_results")